# Lancer le pipeline sur Google Colab

Ce notebook exécute tout le projet (`data`, `classification`, `evaluation`, `retrieval`,
`audit`) sur le GPU de Colab, puis range le résultat sur Google Drive, car la machine Colab est
effacée à la déconnexion.

**Avant de commencer** : *Exécution → Modifier le type d'exécution → GPU (T4)*, puis exécuter
les cellules **dans l'ordre**. Garder l'onglet ouvert pendant l'entraînement.

Durée indicative (estimation, à vérifier dans `run_info.json`) : environ une heure et demie
au total, dont la majeure partie pour l'entraînement.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 1. Réglages et Google Drive

`BRANCH` : branche du dépôt à exécuter (`master` une fois les changements fusionnés).
Si le dépôt est **privé**, créer un jeton GitHub (lecture du dépôt) et l'enregistrer dans les
*Secrets* de Colab (icône clé, à gauche) sous le nom `GITHUB_TOKEN`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

REPO = "https://github.com/KotDAzur/contract-review-nlp.git"
BRANCH = "master"
EXP = "legalbert"
DRIVE_RUNS = "/content/drive/MyDrive/contract-review-nlp/runs"

try:  # dépôt privé : jeton enregistré dans les Secrets de Colab
    from google.colab import userdata

    REPO = REPO.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
except Exception:
    pass  # dépôt public, ou pas de secret

## 2. Installer le projet

`uv sync --locked` installe exactement les versions du fichier `uv.lock`, comme sur les autres
machines. La première installation télécharge PyTorch et les bibliothèques CUDA : quelques
minutes.

In [ ]:
!pip -q install uv
!git clone --depth 1 --branch {BRANCH} {REPO} /content/contract-review-nlp
%cd /content/contract-review-nlp
!uv sync --locked

In [ ]:
# S'arrête avec une erreur claire si le GPU n'est pas utilisable : sans lui, l'entraînement
# prendrait des heures.
!uv run python -c "import torch; assert torch.cuda.is_available(), 'GPU indisponible : changer le type d execution'; print(torch.cuda.get_device_name(0))"

## 3. Restaurer un run précédent

Sans effet au premier lancement. Après une coupure, ré-exécuter les cellules 1 et 2 puis celle-ci,
et reprendre à l'étape qui n'a pas abouti. **Faire la reprise le même jour** (UTC) : le dossier du
run porte la date du jour.

In [ ]:
!mkdir -p {DRIVE_RUNS} artifacts/runs
!cp -r {DRIVE_RUNS}/. artifacts/runs/

## 4. Données et entraînement

La partie la plus longue. À la fin, le run est copié sur Drive : si la suite échoue, le modèle
entraîné n'est pas perdu.

In [ ]:
!uv run main.py run --exp {EXP} --steps data classification
!cp -r artifacts/runs/. {DRIVE_RUNS}/

## 5. Évaluation, recherche et audit

L'évaluation relit le modèle entraîné ; le retrieval indexe les 60 000 clauses (BM25 sur 1 000
requêtes : environ dix minutes) ; l'audit traduit 300 clauses en français.

In [ ]:
!uv run main.py run --exp {EXP} --steps evaluation retrieval audit
!cp -r artifacts/runs/. {DRIVE_RUNS}/

## 6. Lire les résultats

In [ ]:
import glob
import json

from IPython.display import Image, Markdown, display

run = sorted(glob.glob(f"artifacts/runs/*_{EXP}"))[-1]
print("run :", run)
info = json.load(open(f"{run}/run_info.json"))
print("GPU de l'entraînement :", info["steps"]["classification"]["gpu"])

results = json.load(open(f"{run}/metrics/evaluation.json"))
for model in ("baseline", "transformer"):
    test = results[model]["test"]
    print(model, "| accuracy", test["accuracy"]["value"], "| macro-F1", test["macro_f1"]["value"])
print("écart transformer - baseline (macro-F1) :", results["transformer_minus_baseline"]["macro_f1"])
print("seuil de confiance :", results["transformer"]["selective"])

display(Image(f"{run}/figures/weakest_classes.png"))
display(Markdown(open(f"{run}/AUDIT.md", encoding="utf-8").read()))

## 7. Récupérer les résultats sur ton ordinateur

Archive légère (sans le modèle de 440 Mo) : métriques, figures, `AUDIT.md`, config et
`run_info.json`. Le run complet reste sur Google Drive, dans `contract-review-nlp/runs`.

In [ ]:
!cd artifacts/runs && zip -r -q /content/resultats.zip */metrics */figures */AUDIT.md */config.yaml */run_info.json

from google.colab import files

files.download("/content/resultats.zip")